# 딥러닝응용III — Lecture 03 실습
# 토큰화 기초: BPE 어휘 집합 구축과 GPT 입력값 생성 (학생용 워크시트)

| 항목 | 내용 |
|------|------|
| **강의** | 딥러닝응용III (자연어처리) |
| **수업** | 3주차 — 토큰화 기초 |
| **전공** | 데이터사이언스전공, 동덕여자대학교 |
| **교수** | 유원상 교수 |
| **참고** | 이기창, *BERT와 GPT로 배우는 자연어 처리* (이지스퍼블리싱) |

> 이 노트북은 교수가 직접 코드를 설명하는 실습 자료입니다.
> **빈칸**을 채우고, **질문**에 답하면서 코드의 의미를 이해하세요.
> 중간고사 코딩 문제는 이 노트북의 내용을 바탕으로 출제됩니다.

---

### 🎯 학습 목표

1. **서브워드 토큰화의 필요성** 이해 — 단어/문자 단위 토큰화의 한계를 극복하는 원리
2. **BPE 어휘 집합 구축 과정** 이해 — `ByteLevelBPETokenizer`로 `vocab.json`, `merges.txt` 생성
3. **GPT 입력값 생성** — `input_ids`와 `attention_mask`의 의미와 생성 방법 이해
4. **Byte-Level BPE의 한국어 표현 원리** — UTF-8 바이트와 `byte_encoder` 매핑 이해

---

## 셀 1. 실습 환경 만들기

### ⚙️ 런타임 설정

상단 메뉴 → **런타임** → **런타임 유형 변경** → 하드웨어 가속기 → **None (CPU)**

> 이 실습은 토크나이저 학습과 텍스트 전처리만 하므로 GPU가 필요 없습니다.

### 📦 설치하는 패키지 3가지

| 패키지 | 역할 |
|--------|------|
| `korpora` | 한국어 공개 말뭉치 로더. `Korpora.load("nsmc")`로 NSMC를 자동 다운로드 |
| `tokenizers` | HuggingFace 저수준 토크나이저. BPE 어휘 집합 **학습 자체**를 수행 (Rust 구현) |
| `transformers` | HuggingFace 고수준 라이브러리. 학습된 토크나이저를 `GPT2Tokenizer`로 감싸 입력값 생성 |

> 💡 **핵심**: `tokenizers`로 어휘 집합을 **만들고(학습)**, `transformers`로 그 결과를 **사용(로드)**

### ❓ Q1-1. 패키지 역할 구분

> (a) `vocab.json`과 `merges.txt`를 **생성**하는 패키지는? (korpora / tokenizers / transformers)
>
> (b) 위 결과물을 **로드하여 모델 입력값을 만드는** 패키지는?

**답:**



### 📝 Q1-2. 빈칸 채우기 — 패키지 설치

아래 코드의 `______`를 채우세요.

```python
!pip install -q ______ ______ ______     # (a)(b)(c) 세 패키지
```

In [ ]:
# 패키지 설치
!pip install -q ______ ______ ______   # TODO: 세 패키지 이름을 채우세요

import tokenizers, transformers
print(f"tokenizers   버전: {tokenizers.__version__}")
print(f"transformers 버전: {transformers.__version__}")

---

## 셀 2. 구글 드라이브 연동

토크나이저 학습 결과물(`vocab.json`, `merges.txt`)을 **구글 드라이브에 저장**합니다.
- Colab 런타임은 **임시 VM** — 세션 종료 시 모든 파일 삭제
- 구글 드라이브에 저장하면 런타임 종료 후에도 결과 유지
- 다음 실습(Lecture 04)에서 재사용

### ❓ Q2-1. 저장 경로 이해

> (a) `force_remount=True`의 역할은?
>
> (b) `os.makedirs()`에서 `exist_ok=True`를 빼면 어떤 상황에서 에러가 발생하나요?

**답:**



### 📝 Q2-2. 빈칸 채우기 — 드라이브 연동

아래 코드의 `______`를 채우세요.

```python
from google.colab import ______              # (a) 모듈
______.mount('/gdrive', force_remount=True)   # (b) 마운트

import os
BBPE_DIR = "/gdrive/My Drive/nlpbook/bbpe"
os.______(BBPE_DIR, ______=True)              # (c)(d) 함수와 파라미터
```

In [ ]:
from google.colab import ______              # TODO: (a)
______.mount('/gdrive', force_remount=True)   # TODO: (b)
print("✅ 구글 드라이브 연결 완료")

In [ ]:
import os
BBPE_DIR = "/gdrive/My Drive/nlpbook/bbpe"
os.______(BBPE_DIR, ______=True)              # TODO: (c)(d)
print(f"✅ BBPE 저장 경로: {BBPE_DIR}")

### 🐛 Q2-3. 오류 수정 — 드라이브 경로

아래 코드에는 **2개의 오류**가 있습니다. 찾아서 수정하세요.

```python
BBPE_DIR = "/gdrive/nlpbook/bbpe"             # 오류 1
os.makedirs(BBPE_DIR)                          # 오류 2
```

**답:**



---
## 셀 3. 말뭉치 로딩 및 전처리

### 📚 NSMC(Naver Sentiment Movie Corpus)

| 항목 | 내용 |
|------|------|
| 총 데이터 수 | 약 200,000개 (train 150,000 / test 50,000) |
| 레이블 | 긍정(1) / 부정(0) |
| 특징 | 구어체, 비속어, 이모티콘, 오타 등 실제 인터넷 언어 포함 |

### 핵심: 토크나이저 학습은 **비지도(unsupervised)** 방식

텍스트의 통계적 패턴(어떤 글자쌍이 자주 함께 등장하는가)만 학습하므로 레이블이 **불필요**합니다.

### ❓ Q3-1. 말뭉치 이해

> (a) `nsmc.train[0].text`와 `nsmc.train[0].label`은 각각 무엇을 반환하나요?
>
> (b) 토크나이저 학습에 레이블(0/1)이 필요 없는 이유를 한 문장으로 설명하세요.

**답:**



### 📝 Q3-2. 빈칸 채우기 — 말뭉치 로딩

아래 코드의 `______`를 채우세요.

```python
from ______ import ______                     # (a)(b) 패키지와 클래스
nsmc = ______.load("______", force_download=True)  # (c)(d)
print(f"학습 데이터: {len(nsmc.______):,}개")  # (e)
```

In [ ]:
from ______ import ______                     # TODO: (a)(b)

print("NSMC 데이터를 다운로드합니다...")
nsmc = ______.load("______", force_download=True)  # TODO: (c)(d)

print(f"\n✅ 학습 데이터: {len(nsmc.______):,}개")  # TODO: (e)
print(f"✅ 테스트 데이터: {len(nsmc.test):,}개")

print("\n[데이터 예시 3개]")
for i in range(3):
    item = nsmc.train[i]
    label_str = "긍정" if item.label == 1 else "부정"
    print(f"  [{i}] ({label_str}) {item.text}")

---

### 💾 텍스트 파일로 저장하는 이유

`ByteLevelBPETokenizer.train()`은 **파일 경로 목록**(`files=[...]`)을 입력받습니다.
메모리의 리스트가 아닌 **디스크 파일**을 직접 읽어서 학습합니다. 저장 형식: 한 줄에 문장 하나.

### 📝 Q3-3. 빈칸 채우기 — 텍스트 저장 함수

아래 코드의 `______`를 채우세요.

```python
def write_lines(path, lines):
    with open(path, '______', encoding='______') as f:    # (a) 모드 (b) 인코딩
        for line in lines:
            f.write(f'{line}______')                       # (c) 줄 구분자

write_lines(TRAIN_PATH, nsmc.______.______)  # (d)(e) 속성.메서드
```

In [ ]:
import os

def write_lines(path, lines):
    with open(path, '______', encoding='______') as f:    # TODO: (a)(b)
        for line in lines:
            f.write(f'{line}______')                       # TODO: (c)

TRAIN_PATH = "/root/train.txt"
TEST_PATH  = "/root/test.txt"

write_lines(TRAIN_PATH, nsmc.______.______)  # TODO: (d)(e)
write_lines(TEST_PATH,  nsmc.______.______)  # TODO: (f)(g)

train_mb = os.path.getsize(TRAIN_PATH) / (1024 * 1024)
test_mb  = os.path.getsize(TEST_PATH)  / (1024 * 1024)
print(f"✅ train.txt 저장 완료 ({train_mb:.1f} MB)")
print(f"✅ test.txt  저장 완료 ({test_mb:.1f} MB)")

### 🐛 Q3-4. 오류 수정 — 데이터 저장

아래 코드에는 **3개의 오류**가 있습니다. 찾아서 수정하세요.

```python
def write_lines(path, lines):
    with open(path, 'r', encoding='utf-8') as f:       # 오류 1
        for line in lines:
            f.write(f'{line}')                          # 오류 2

write_lines(TRAIN_PATH, nsmc.train.get_all_labels())   # 오류 3
```

**답:**



### 🧮 Q3-5. 데이터 크기 계산

> (a) `nsmc.train.get_all_texts()`가 반환하는 리스트의 길이는?
>
> (b) train.txt + test.txt에 총 몇 줄이 저장되나요?

**답:**



---
## 셀 4. BBPE 어휘 집합 구축 (GPT 토크나이저)

### 🔬 Byte-Level BPE(BBPE)란?

| 특징 | 일반 BPE | BBPE |
|------|---------|------|
| 초기 어휘 크기 | 사용 문자 수 (한글 ~11,172) | **256개** (0x00~0xFF) |
| OOV 문제 | 있음 | **완전히 없음** |
| 사용 모델 | - | GPT-2, GPT-3, RoBERTa 등 |

### vocab.json vs merges.txt

| 파일 | 역할 |
|------|------|
| `vocab.json` | 완성된 어휘 집합. 토큰 → 인덱스 매핑 |
| `merges.txt` | 병합 규칙 목록. 토큰화 시 바이트 병합 순서 결정 |

> ⚠️ **두 파일 모두** 있어야 토크나이저 재로딩 가능

### ❓ Q4-1. BBPE 개념 이해

> (a) 한글 "가"는 UTF-8에서 몇 바이트?
>
> (b) BBPE의 초기 어휘가 256개인 이유?
>
> (c) BBPE에서 OOV가 없는 이유?

**답:**



### 📝 Q4-2. 빈칸 채우기 — BBPE 학습 코드

아래 코드의 `______`를 채우세요.

```python
from tokenizers import ______Tokenizer     # (a)
bbpe_tokenizer = ______Tokenizer()         # (b)
bbpe_tokenizer.______(                     # (c) 학습 메서드
    files=[TRAIN_PATH, ______],             # (d)
    vocab_size=______,                       # (e)
    min_frequency=______,                    # (f)
    special_tokens=["______"],               # (g)
)
bbpe_tokenizer.______(BBPE_DIR)             # (h) 저장 메서드
```

In [ ]:
from tokenizers import ______Tokenizer     # TODO: (a)

bbpe_tokenizer = ______Tokenizer()         # TODO: (b)

print("🔄 BBPE 토크나이저 학습 중... (약 1~2분)")
bbpe_tokenizer.______(                     # TODO: (c)
    files=[TRAIN_PATH, ______],             # TODO: (d)
    vocab_size=______,                       # TODO: (e)
    min_frequency=______,                    # TODO: (f)
    special_tokens=["______"],               # TODO: (g)
)

bbpe_tokenizer.______(BBPE_DIR)             # TODO: (h)

print(f"\n✅ 저장 완료: {BBPE_DIR}")
for fname in sorted(os.listdir(BBPE_DIR)):
    fpath = os.path.join(BBPE_DIR, fname)
    print(f"   📄 {fname}  ({os.path.getsize(fpath):,} bytes)")

### 🐛 Q4-3. 오류 수정 — BBPE 학습

아래 코드에는 **4개의 오류**가 있습니다. 찾아서 수정하세요.

```python
from tokenizers import ByteLevelBPETokenizer

bbpe = ByteLevelBPETokenizer()
bbpe.train(
    files="train.txt",                    # 오류 1
    vocab_size=10000,
    min_frequency=0,                      # 오류 2
    special_tokens="[PAD]",               # 오류 3
)
bbpe.save("/root/output")                 # 오류 4
```

**답:**



### 🧮 Q4-4. 병합 횟수 계산

> (a) `vocab_size=10000`, `special_tokens=["[PAD]"]`이면 BPE 병합은 최대 몇 회?
> (힌트: 초기 256바이트 + 특수토큰 1개 + 병합 N회 = 10,000)
>
> (b) `[PAD]`의 인덱스는 몇 번?

**답:**



---

### 🔍 BBPE 토큰 표시 방식 이해하기 (매우 중요!)

`vocab.json`을 열어보면 한국어가 **깨진 문자처럼** 보입니다 — 이것은 오류가 아닙니다!

```
"안" → UTF-8: [0xEC, 0x95, 0x88]  (3바이트)
     → byte_encoder: 각 바이트를 유니코드 문자 1개로 매핑
     → BBPE 내부 표현: "ìĿ¨" (3개의 특수 문자)
```

`decode_bbpe_token()` 함수가 이 과정을 **역방향**으로 수행하여 한국어를 복원합니다.

| 함수 | 역할 |
|------|------|
| `bytes_to_unicode()` | 바이트(0~255) → 유니코드 문자 매핑 딕셔너리 생성 |
| `byte_encoder` | 위 함수의 결과. **바이트→유니코드** 방향 |
| `byte_decoder` | 역방향. **유니코드→바이트** 방향 |
| `decode_bbpe_token(token)` | BBPE 내부 표현 → 한국어 복원 |

### ❓ Q5-1. byte_encoder 이해

> (a) `byte_encoder`는 어떤 방향? (바이트→유니코드 / 유니코드→바이트)
>
> (b) `byte_decoder`는?
>
> (c) 한글 1글자 = UTF-8 3바이트 → BBPE 내부에서 유니코드 문자 몇 개?

**답:**



### 📝 Q5-2. 빈칸 채우기 — decode_bbpe_token 핵심 로직

아래 함수 핵심부의 `______`를 채우세요.

```python
def decode_bbpe_token(token):
    byte_list = [______(c) for c in token]           # (a) 어떤 딕셔너리?
    decoded = ______(byte_list).decode("utf-8",       # (b) 바이트 변환 함수
                                       errors="replace")
    if '______' in decoded:                           # (c) 불완전 UTF-8 표시 문자
        return '[hex]'
    return decoded
```

In [ ]:
# 헬퍼 함수 정의
import json

def bytes_to_unicode():
    bs = (list(range(ord("!"), ord("~") + 1))
          + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

def decode_bbpe_token(token: str) -> str:
    try:
        byte_list = [______(c) for c in token]           # TODO: (a)
    except KeyError:
        return token
    decoded = ______(byte_list).decode("utf-8", errors="replace")  # TODO: (b)
    if '______' in decoded:                               # TODO: (c)
        return '[' + ' '.join(f'{b:02x}' for b in byte_list) + ']'
    return decoded.replace(' ', '▁')

def decode_merge_pair(part_a, part_b):
    return decode_bbpe_token(part_a + part_b)

# vocab.json 확인
with open(f"{BBPE_DIR}/vocab.json", 'r', encoding='utf-8') as f:
    vocab = json.load(f)
print(f"=== vocab.json: 총 {len(vocab):,}개 ===")
for i, (token, idx) in enumerate(vocab.items()):
    if i >= 8: break
    print(f"  {idx:5d}  {repr(token):20}  {decode_bbpe_token(token)}")

### 🐛 Q5-3. 오류 수정 — byte_encoder 관련

아래 코드에는 **2개의 오류**가 있습니다. 찾아서 수정하세요.

```python
byte_encoder = bytes_to_unicode()
byte_decoder = byte_encoder                          # 오류 1

def decode_bbpe_token(token):
    byte_list = [byte_encoder[c] for c in token]     # 오류 2
    return bytes(byte_list).decode("utf-8", errors="replace")
```

**답:**



### ❓ Q5-4. merges.txt 해석

> (a) merges.txt에서 초기 병합이 hex로 표시되는 이유?
>
> (b) 병합이 진행될수록 한국어로 표시되는 이유?

**답:**



In [ ]:
# BBPE 동작 확인 — 바이트 인코딩 원리
sample = "안녕하세요 반갑습니다"

bytes_repr = ' '.join([f'{b:02x}' for b in sample.encode('utf-8')])
print(f"원문       : {sample}")
print(f"UTF-8 바이트: {bytes_repr}")
print(f"글자 수    : {len(sample)}자  →  바이트 수: {len(sample.encode('utf-8'))}바이트")

result = bbpe_tokenizer.encode(sample)
readable_tokens = [decode_bbpe_token(t) for t in result.tokens]

print(f"\n[ BBPE 토큰화 결과 ]")
for pos, (raw, readable) in enumerate(zip(result.tokens, readable_tokens), 1):
    print(f"  {pos:4d}   {raw:20}   {readable}")
print(f"\n  → 바이트 {len(sample.encode('utf-8'))}개가 {len(result.tokens)}개 토큰으로 병합됨")

### 🧮 Q5-5. 바이트 수 계산

> (a) "안녕하세요" (5글자, 한글) → UTF-8 몇 바이트?
>
> (b) "Hello" (5글자, 영문) → UTF-8 몇 바이트?
>
> (c) "딥러닝 Deep" → 총 UTF-8 바이트 수? (공백 1바이트)

**답:**



---
## 셀 5. GPT 입력값 만들기

| 입력값 | 설명 |
|--------|------|
| `input_ids` | 토큰 인덱스 시퀀스. 각 토큰 → 어휘 집합의 인덱스(정수) |
| `attention_mask` | 실제 토큰=`1`, 패딩 토큰=`0`. 모델이 패딩을 무시하도록 알려줌 |

| 개념 | 동작 |
|------|------|
| 패딩(padding) | 짧은 문장 뒤에 `[PAD]` 추가하여 길이 맞춤 |
| 잘림(truncation) | `max_length` 초과 부분 잘라냄 |

### 📝 Q6-1. 빈칸 채우기 — GPT 토크나이저 로드

아래 코드의 `______`를 채우세요.

```python
from transformers import ______                        # (a) 클래스
tokenizer_gpt = ______.from_pretrained(BBPE_DIR)       # (b) 클래스
tokenizer_gpt.______ = "______"                        # (c)(d) 속성=값
```

In [ ]:
from transformers import ______                        # TODO: (a)

tokenizer_gpt = ______.from_pretrained(BBPE_DIR)       # TODO: (b)
tokenizer_gpt.______ = "______"                        # TODO: (c)(d)

print(f"✅ GPT 토크나이저 로드 완료")
print(f"   어휘 집합 크기 : {len(tokenizer_gpt):,}개")
print(f"   패딩 토큰      : '{tokenizer_gpt.pad_token}' (id={tokenizer_gpt.pad_token_id})")

### 🐛 Q6-2. 오류 수정 — 토크나이저 로드

아래 코드에는 **3개의 오류**가 있습니다.

```python
from transformers import BertTokenizer                  # 오류 1

tokenizer = BertTokenizer.from_pretrained(BBPE_DIR)     # 오류 2
# pad_token 설정 없이 바로 배치 인코딩                      # 오류 3 (누락)
batch = tokenizer(["테스트"], padding="max_length", max_length=8)
```

**답:**



### ❓ Q6-3. GPT2Tokenizer 이해

> (a) `from_pretrained()`가 읽는 파일 2개는?
>
> (b) GPT2Tokenizer에 기본적으로 `pad_token`이 없는 이유?

**답:**



In [ ]:
# 예시 문장 토큰화
sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
]

print("=== GPT 토크나이저 토큰화 결과 ===")
for i, sent in enumerate(sentences, 1):
    tokens  = tokenizer_gpt.tokenize(sent)
    readable = [decode_bbpe_token(t) for t in tokens]
    print(f"\n[문장{i}] {sent}")
    print(f"  토큰 수: {len(tokens)}")
    print(f"  한국어 : {readable}")

### 🧮 Q6-4. 토큰화 결과 해석

> 위 출력을 보고 답하세요.
>
> (a) 문장1의 토큰 수는?
>
> (b) 문장3의 토큰 수는?
>
> (c) 세 문장 중 `max_length=12`로 잘림(truncation)이 발생하는 문장은?

**답:**



### 📝 Q6-5. 빈칸 채우기 — 배치 인코딩

아래 코드의 `______`를 채우세요.

```python
batch_gpt = tokenizer_gpt(
    ______,                    # (a) 문장 리스트 변수
    padding="______",          # (b) 패딩 방식
    max_length=______,         # (c) 최대 토큰 수
    truncation=______,         # (d) 잘림 허용 여부
)
```

In [ ]:
batch_gpt = tokenizer_gpt(
    ______,                    # TODO: (a)
    padding="______",          # TODO: (b)
    max_length=______,         # TODO: (c)
    truncation=______,         # TODO: (d)
)

### 🐛 Q6-6. 오류 수정 — 배치 인코딩

아래 코드에는 **2개의 문제점**이 있습니다.

```python
batch = tokenizer_gpt(
    sentences,
    padding="longest",                    # 문제 1
    max_length=12,
    truncation=False,                     # 문제 2
)
```

**답:**



In [ ]:
# GPT 모델 입력 생성 (정상 코드)
batch_gpt = tokenizer_gpt(
    sentences,
    padding="max_length",
    max_length=12,
    truncation=True,
)

header = "구분    " + "  ".join([f"토큰{i:02d}" for i in range(1, 13)])
sep    = "─" * 82

print("=== GPT input_ids ===")
print("  0 = [PAD] 토큰")
print(); print(header); print(sep)
for i, ids in enumerate(batch_gpt['input_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ids]))

print()
print("=== GPT attention_mask ===")
print("  1 = 실제 토큰,  0 = 패딩")
print(); print(header); print(sep)
for i, mask in enumerate(batch_gpt['attention_mask'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in mask]))

### 🧮 Q6-7. input_ids와 attention_mask 해석

> 위 출력을 보고 답하세요.
>
> (a) 문장2의 attention_mask에서 `1`의 개수가 12개이면 무슨 뜻?
>
> (b) 문장3의 실제 토큰이 4개이면 attention_mask의 `0`은 몇 개?

**답:**



### 📝 Q6-8. 빈칸 채우기 — attention_mask 직접 작성

> `max_length=8`, 문장의 실제 토큰 수가 5개일 때:
>
> input_ids =       `[234, 567, 89, 1023, 456, ______, ______, ______]`  (a)(b)(c)
>
> attention_mask =  `[______, ______, ______, ______, ______, ______, ______, ______]`  (d)

### 📝 Q6-8. 빈칸 채우기 — attention_mask 직접 작성

> `max_length=8`, 문장의 실제 토큰 수가 5개일 때:
>
> input_ids =       `[234, 567, 89, 1023, 456, ______, ______, ______]`  (a)(b)(c)
>
> attention_mask =  `[______, ______, ______, ______, ______, ______, ______, ______]`  (d) 8개 모두

**답:**



### 🧮 Q6-9. Shape 계산

> 문장 3개, `max_length=12` 배치 인코딩.
>
> (a) `input_ids`의 shape = ( ______ × ______ )
>
> (b) `max_length=20`, 문장 5개이면 shape = ( ______ × ______ )

**답:**



---
## 📝 강의 요약

| 항목 | 내용 |
|------|------|
| **서브워드 토큰화** | 어휘 집합 크기를 적절히 유지하면서 OOV 문제 방지 |
| **BPE** | 가장 자주 등장하는 바이그램 쌍을 **빈도** 기준으로 반복 병합 |
| **BBPE** | UTF-8 **바이트** 단위에서 시작. OOV 완전 제거. GPT 계열 사용 |
| **vocab.json** | 완성된 어휘 집합. 토큰 → 인덱스 |
| **merges.txt** | 병합 규칙. 토큰화 시 병합 순서 결정 |
| **byte_encoder** | 바이트(0~255) → 유니코드 문자 매핑 |
| **GPT 입력** | `input_ids` + `attention_mask` |

---
## 📋 종합 퀴즈 (중간고사 대비)

---

### Quiz 1. 토큰화 수준 비교

> 빈칸을 채우세요.
>
> | 토큰화 수준 | 어휘 집합 크기 | OOV 문제 |
> |------------|-------------|---------|
> | 단어 단위 | ______ | ______ |
> | 문자 단위 | ______ | ______ |
> | 서브워드 단위 | ______ | ______ |

---

### Quiz 2. BPE 수동 실행

> 초기 어휘: {a, b, c, d}. 빈도표: (a,b,c)→5, (a,b,d)→3, (b,c,d)→2
>
> (a) 모든 바이그램 쌍과 빈도를 나열하세요.
>
> (b) 가장 자주 등장한 쌍은? → 병합 후 새 어휘 집합은?

**답:**

---

### Quiz 3. 병합 횟수

> `vocab_size=10000`, `special_tokens=["[PAD]"]`일 때 BPE 병합 최대 횟수 = ______

---

### Quiz 4. UTF-8 바이트

> (a) "딥러닝"(3글자) → ______ 바이트
>
> (b) "Deep"(4글자) → ______ 바이트
>
> (c) "딥러닝 Deep" → ______ 바이트

---

### Quiz 5. 빈칸 채우기 — 전체 파이프라인

> ```python
> from tokenizers import ______                    # (a)
> from transformers import ______                   # (b)
>
> tok = ______()                                    # (c)
> tok.______(files=["train.txt"], vocab_size=5000,  # (d)
>            special_tokens=["[PAD]"])
> tok.______(BBPE_DIR)                              # (e)
>
> gpt = ______.from_pretrained(BBPE_DIR)            # (f)
> gpt.______ = "[PAD]"                              # (g)
>
> batch = gpt(["안녕"], padding="max_length",
>             max_length=8, ______=True)            # (h)
> ```

---

### Quiz 6. 오류 수정 — 통합

> ```python
> from tokenizers import ByteLevelBPETokenizer
> from transformers import BertTokenizer             # ①
> bbpe = ByteLevelBPETokenizer()
> bbpe.train(files="data.txt",                       # ②
>            special_tokens="[PAD]")                  # ③
> bbpe.save("/output")                               # ④
> tok = BertTokenizer.from_pretrained("/output")      # ⑤
> result = tok(["테스트"], padding="max_length",
>              max_length=10)                         # ⑥ (누락된 것은?)
> ```
>
> 오류 ①~⑥을 모두 수정하세요.

**답:**

---

### Quiz 7. attention_mask

> `max_length=10`, 실제 토큰 6개 → attention_mask = [______ × 10]

---

### Quiz 8. 파일 역할

> (a) merges.txt를 삭제하면?
>
> (b) merges.txt 1순위 = 가장 ______(자주/드물게) 등장한 쌍

---

### Quiz 9. byte_encoder

> (a) key 개수 = ______
>
> (b) 공백(0x20) → BBPE 내부 문자 = ______

---

### Quiz 10. 파이프라인 순서

> ① attention_mask  ② BBPE train  ③ NSMC 다운로드  ④ vocab.json 저장
> ⑤ GPT2Tokenizer 로드  ⑥ input_ids 변환  ⑦ tokenize  ⑧ padding
>
> 순서: __ → __ → __ → __ → __ → __ → __ → __

---

### Quiz 11. 결과 추론

> ```python
> result = tokenizer_gpt(["가나다"], padding="max_length",
>                        max_length=5, truncation=True)
> print(len(result['input_ids'][0]))       # (a) ______
> print(result['attention_mask'][0][-1])    # (b) ______
> ```

---

### Quiz 12. BPE vs WordPiece

> (a) BPE 병합 기준: ______
>
> (b) WordPiece 병합 기준: ______
>
> (c) GPT → ______,  BERT → ______